# Experimentos en Kaggle: CodeLlama-7B-Instruct a 16, 8 y 4 bits

**Antes de ejecutar**, en el panel derecho (*Settings / Session options*):
- **Accelerator: GPU T4 x2** (se usa solo una T4, igual que en Colab)
- **Internet: On** (requiere cuenta verificada con teléfono)

Luego **Save Version → Save & Run All (Commit)**. Corre en segundo plano (~3-4 h; límite 12 h),
puedes cerrar el navegador. Al terminar, en la pestaña *Output* descarga `resultados_kaggle.zip`.

Cada paso se detiene con error si algo falla, para no gastar horas de GPU en vano.

In [ ]:
import os, subprocess, sys, json
# Una sola T4, como en Colab (con dos, device_map='auto' repartiria el modelo y la medicion de memoria seria otra)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTHONUNBUFFERED'] = '1'

def sh(cmd, cwd=None):
    """Ejecuta un comando mostrando su salida; lanza error si falla."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for linea in p.stdout:
        if 'Loading weights' in linea:   # barra de progreso muy larga
            continue
        print(linea, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'Fallo ({p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
print(sys.version)

In [ ]:
W = '/kaggle/working'
REPO = f'{W}/repo'
EXP = f'{REPO}/experimentos'
SCR = f'{EXP}/scripts'
sh(f'rm -rf {REPO} && git clone -q -b experimentos-diseno https://github.com/CristhianBravo865/Articulo_Construccion_Software.git {REPO}')
sh(f'pip install -q -r {EXP}/requirements.txt')
sh('pip install -q --no-deps codebleu==0.7.0')
sh(f'{sys.executable} -m bandit --version')
sh(f'{sys.executable} -c "import codebleu, bandit, bitsandbytes, transformers; print(\'codebleu, bandit, bnb OK; transformers\', transformers.__version__)"')

## Preparar los datos
Descarga FixMe (815 MB, Zenodo, a `/tmp` para no llenar el Output) y SecurityEval.
El embudo debe dar E1_muestra 150 y E2_con_hallazgo_bandit 49 (igual que en la laptop).

In [ ]:
sh('wget -q -O /tmp/FixMe-v1.db "https://zenodo.org/records/10955342/files/FixMe-v1.db?download=1"')
sh('md5sum /tmp/FixMe-v1.db')   # esperado: fd5de8fd5f9f85f380787a641b72182e
sh(f'{sys.executable} 01_preparar_dataset.py --fixme /tmp/FixMe-v1.db', cwd=SCR)
embudo = json.load(open(f'{EXP}/datos/embudo.json'))
print(embudo)
assert embudo['E1_muestra'] == 150 and embudo['E2_con_hallazgo_bandit'] == 49, 'El embudo no coincide con la laptop'
os.remove('/tmp/FixMe-v1.db')

## Generar parches
Primero E2 (49 casos), luego E1 (150 casos).

In [ ]:
for exp in ['E2', 'E1']:
    for cfg in ['L-16', 'L-8', 'L-4']:
        sh(f'{sys.executable} 02_generar_parches.py --experimento {exp} --config {cfg} --hardware kaggle-t4', cwd=SCR)

## Medición controlada de costo (RQ3)
Mismos 10 prompts, 128 tokens exactos, 3 repeticiones por configuración.

In [ ]:
for cfg in ['L-16', 'L-8', 'L-4']:
    sh(f'{sys.executable} 05_medir_costo.py --config {cfg} --hardware kaggle-t4', cwd=SCR)

## Evaluar, resumir y empaquetar

In [ ]:
sh(f'{sys.executable} 03_evaluar.py ../resultados/gen_*_kaggle-t4.jsonl', cwd=SCR)
sh(f'{sys.executable} 04_analizar.py', cwd=SCR)
sh(f'cd {EXP} && zip -q -r {W}/resultados_kaggle.zip resultados datos/embudo.json')
sh(f'rm -rf {REPO}')   # en Output queda solo el zip
sh(f'ls -la {W}')